In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import pickle
import numpy

In [ ]:
df_embeddings = pd.read_csv("/content/embeddings.csv")
df_labels = pd.read_csv("/content/PCA_transformed_final_data.csv")

In [ ]:
df_embeddings.drop(columns=["Unnamed: 0"], inplace=True)

pca = PCA(n_components=10, random_state=42)

pca.fit(df_embeddings)

pickle.dump(  pca, open("pca.pkl", "wb"))

df_embeddings = pca.transform(df_embeddings)

In [ ]:
df_final = pd.DataFrame(df_embeddings, columns=[f'pca_{i}' for i in range(df_embeddings.shape[1])])
df_final['Host'] = df_labels['Host']
display(df_final.head())

,pca_0,pca_1,pca_2,pca_3,pca_4,pca_5,pca_6,pca_7,pca_8,pca_9,Host
0,0.106643,-0.107236,0.019642,0.017026,0.028785,-0.015735,0.030059,0.025177,-0.038012,-0.000481,Homo sapiens
1,0.119046,-0.114722,0.018019,0.016784,0.029485,-0.015730,0.033407,0.024484,-0.035320,0.001481,Homo sapiens
2,0.113494,-0.112023,0.018894,0.016444,0.030118,-0.017464,0.034981,0.024929,-0.033272,-0.000164,Homo sapiens
3,0.147456,-0.105220,0.014620,0.009625,0.034620,-0.019452,0.031787,0.023306,-0.018862,0.000944,Canis lupus familiaris
4,0.122027,-0.111845,0.017665,0.016176,0.025689,-0.014403,0.032327,0.024245,-0.035777,0.001707,Canis lupus familiaris


In [ ]:
from re import X
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


df_final.dropna(subset=['Host'], inplace=True)

X = df_final.drop(columns=['Host'])
y = df_final['Host']

X = X.dropna()

LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier()

model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

                        precision    recall  f1-score   support

            Bos taurus       0.88      0.63      0.74       117
               Canidae       0.61      0.37      0.46        30
Canis lupus familiaris       0.87      0.96      0.91       730
      Eptesicus fuscus       0.86      0.86      0.86        22
           Felis catus       0.33      0.11      0.17        27
          Homo sapiens       0.84      0.31      0.46        51
              Melogale       0.93      1.00      0.96        25
            Mephitidae       0.73      0.61      0.66       109
         Procyon lotor       0.73      0.92      0.82       135
                Vulpes       0.80      0.81      0.81       107

              accuracy                           0.83      1353
             macro avg       0.76      0.66      0.68      1353
          weighted avg       0.82      0.83      0.82      1353



In [ ]:
pickle.dump(model, open("model.pkl", "wb"))

In [29]:
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to(device)
model.eval()

def embed_sequence(sequence, batch_size=16):

    sequences = [sequence]
    embeddings = []

    for i in tqdm(range(0, len(sequences), batch_size), desc="Embedding"):
        batch_seqs = [" ".join(list(seq)) for seq in sequences[i:i+batch_size]]
        tokens = tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=524
        ).to(device)

        with torch.no_grad():
            output = model(**tokens)

        batch_emb = output.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.extend(batch_emb)


    return embeddings[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

In [37]:
def preprocess(seq):
  embeddings = embed_sequence(seq)
  embeddings = embeddings.reshape(1, -1)

  return pca.transform(embeddings)

seq= "MIPQVLLFVPLLVFSSCFGKFPIYTIPDKLGPWSPIDIHHLSCPNNLVVEDEGCTNLSGFSYMELKVGYISAIKVNGFTCTGVVTEAETYTNFVGYVTTTFKRKHFRPTPDACRAAYNWKMAGDPRYEESLHNPYPDYHWLRTVKTSKESLVIISPSVADLDPYDKSLHSRVFPSGKCSGITVSSTCCSTNHDYTIWMPENPRLGTSCDIFTNSRGKRASKGGKTCGFVDERGLYKSLKGACKLKLCGVLGLRLMDGTWVAIQTSDEIKWCSPDQLVNLHDFHSDEIEHLVVEELVKKREECLDALETIMTTKSVSFRRLSHLRKLVPGFGKAYTIFNKTLMEADAHYKSIRTWNEIIPSKGCLRVGGRCHPHVNGVFFNGIILGPDGHVLIPEMQSSLLHQHMELLESSVIPLMHPLADPSTVFKDGDEAEDFVEHHLPDVHKQISGVDLGLPNWGKYVLVSAGALTALMLMIFLMTCCRKTNRAESIQHSPGETGRKVSVTSHNGRVISSWESYKRGGETKL"

arr = preprocess(seq)

loaded_model = pickle.load(open("/content/randforest_for_saved_pca.pkl", "rb"))

print(loaded_model.predict_proba(arr))

Embedding: 100%|██████████| 1/1 [00:07<00:00,  7.51s/it]

[[0.         0.         0.92466667 0.         0.         0.07533333
  0.         0.         0.         0.        ]]



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
